In [ ]:
import numpy as np
import pymc as pm
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import os
import random
import xarray as xr

from scipy.stats import norm, halfnorm, percentileofscore

In [ ]:
def compute_experienced_thi(thi_values, decay_lambda=9.99):
    experienced_thi = np.zeros_like(thi_values)
    for t in range(len(thi_values)):
        weights = np.exp(-decay_lambda * (t - np.arange(t+1)))
        experienced_thi[t] = np.sum(thi_values[:t+1] * weights)
    return experienced_thi

In [ ]:
def prepare_train_df(df, start_date, end_date):
    dfs = []
    i = 0
    for cycle_in in [3,4]:
        for idx in df['Animal Name'].unique():
            sel = (
            (df['Animal Name'] == idx) &
            (df['Lactation Number']==cycle_in) &
            (df['Lactation Days'] < 305) &
            (df['Lactation Days'] > 1) &
            (df['Day Production'] > 15) &
            (~df['Day Production'].isna()) &
            (df['DateTime'] > pd.to_datetime(start_date)) &
            (df['DateTime'] < pd.to_datetime(end_date))
            )
            df2 = df.loc[sel,['DateTime','Lactation Days','Day Production', 'Daily THI Degree']].drop_duplicates() #.groupby(['DateTime','Lactation Days'],as_index=False).sum()
            if len(df2)<=150: continue
            df2['Daily THI Degree'] = df2['Daily THI Degree']
            df2['cumsum_THI'] = (df2['Daily THI Degree']).cumsum()
            df2['weighted_THI'] = compute_experienced_thi(df2['Daily THI Degree'])
            df2['idx'] = i
            df2['name'] = df.loc[sel,'Animal Name'].iloc[0]
            i += 1
            dfs.append(df2)
    df2 = pd.concat(dfs)
    
    #df2 = add_prev_thi(df2,df_prev_weather)
    
    y = df2['Day Production'].values
    thi = df2['weighted_THI'].values
    t = df2['Lactation Days'].values
    idx = df2['idx'].values
    n_groups = len(np.unique(idx))
    return df2, y, thi, t, idx, n_groups

In [ ]:
def run_model(y, thi, t, idx, n_groups, num_samples=3000, num_chains=4):

    with pm.Model() as model:

        # --- Hyperpriors ---
        mu_log_a = pm.Normal("mu_log_a", mu=3.5, sigma=0.5)
        sigma_log_a = pm.HalfNormal("sigma_log_a", sigma=0.3)
    
        mu_b = pm.Normal("mu_b", mu=0.2, sigma=0.1)
        sigma_b = pm.HalfNormal("sigma_b", sigma=0.1)
    
        mu_c = pm.HalfNormal("mu_c", sigma=0.05)
        sigma_c = pm.HalfNormal("sigma_c", sigma=0.01)

        #mu_beta = pm.Normal("mu_beta", mu=0, sigma=0.1)
        #sigma_beta = pm.HalfNormal("sigma_beta", sigma=0.05)
    
        mu_gamma = pm.Normal("mu_gamma", mu=0, sigma=0.01)
        sigma_gamma = pm.HalfNormal("sigma_gamma", sigma=0.005)
    
        # --- Group-level parameters (non-centered) ---
        log_a_offset = pm.Normal("log_a_offset", mu=0, sigma=1, shape=n_groups)
        log_a_group = pm.Deterministic("log_a_group", mu_log_a + log_a_offset * sigma_log_a)
    
        b_offset = pm.Normal("b_offset", mu=0, sigma=1, shape=n_groups)
        b_group = pm.Deterministic("b_group", mu_b + b_offset * sigma_b)
    
        c_offset = pm.Normal("c_offset", mu=0, sigma=1, shape=n_groups)
        c_group = pm.Deterministic("c_group", mu_c + c_offset * sigma_c)
    
        gamma_offset = pm.Normal("gamma_offset", mu=0, sigma=1, shape=n_groups)
        gamma_group = pm.Deterministic("gamma_group", mu_gamma + gamma_offset * sigma_gamma)

        #beta_offset = pm.Normal("beta_offset", mu=0, sigma=1, shape=n_groups)
        #beta_group = pm.Deterministic("beta_group", mu_beta + beta_offset * sigma_beta)
    
        # --- Observation noise ---
        sigma = pm.HalfNormal("sigma", sigma=1)
    
        # --- Model prediction ---
        log_a = log_a_group[idx]
        b = b_group[idx] #+ beta_group[idx] * thi
        c = c_group[idx] + gamma_group[idx] * thi  # group-specific gamma applied
    
        log_mu = log_a + b * pm.math.log(t) - c * t
    
        # --- Likelihood ---
        y_obs = pm.StudentT("y_obs", nu=2, mu=log_mu, sigma=sigma, observed=np.log(y))
    
        # --- Sampling ---
        trace = pm.sample(4000, tune=2000, chains=4, target_accept=0.95, progressbar=True)
    
        # --- Posterior predictive ---
        posterior_predictive = pm.sample_posterior_predictive(trace)

    return trace, posterior_predictive

In [ ]:
def load_or_run_model(model_label, build_fn, *args, **kwargs):
    """
    Load trace and posterior predictive if they exist, otherwise run model-building function and save them.

    Parameters:
        model_label (str): A descriptive label like 'woods_base__before__v0'.
        build_fn (callable): Function that returns (model, trace, posterior_predictive).
        *args, **kwargs: Arguments passed to build_fn.

    Returns:
        model, trace, posterior_predictive
    """
    trace_path = f"model_traces/{model_label}__trace.nc"
    ppc_path = f"model_traces/{model_label}__ppc.nc"
    trace_exists = os.path.exists(trace_path)
    ppc_exists = os.path.exists(ppc_path)
    if trace_exists and ppc_exists:
        print(f"Loading existing trace and posterior predictive for '{model_label}'")
        trace = az.from_netcdf(trace_path)
        posterior_predictive = az.from_netcdf(ppc_path)
    else:
        print(f"Running model for '{model_label}'...")
        trace, posterior_predictive = build_fn(*args, **kwargs)
        #trace, posterior_predictive = run_model(y, thi, t, idx, n_groups)
        az.to_netcdf(trace, trace_path)
        az.to_netcdf(posterior_predictive, ppc_path)
    return trace, posterior_predictive

In [ ]:
# def compute_experienced_thi(thi_values, decay_lambda=0.05):
#     experienced_thi = np.zeros_like(thi_values)
#     for t in range(len(thi_values)):
#         weights = np.exp(-decay_lambda * (t - np.arange(t+1)))
#         experienced_thi[t] = np.sum(thi_values[:t+1] * weights)
#     return experienced_thi

In [ ]:
# plt.plot(compute_experienced_thi(df_before[df_before['idx']==40]['Daily THI Degree']))
# plt.plot(df_before[df_before['idx']==0]['Daily THI Degree'])
#plt.plot(df_before[df_before['idx']==0]['weighted_THI'])

In [ ]:
# df_prev_weather = pd.read_csv('data/THI_degree_per_day.csv')
# df_prev_weather['Daily THI Degree'] = df_prev_weather['Daily THI Degree']
df = pd.read_csv('data/MasterData_final_THI_degree.csv', low_memory=False)
df = df[~df['Animal Name'].isna()]
df.sort_values(by='DateTime', inplace=True)
df['month_name'] = pd.to_datetime(df['DateTime']).dt.month_name()
df['month'] = pd.to_datetime(df['DateTime']).dt.month
df['DateTime'] = pd.to_datetime(df['DateTime'])  # do this once

In [ ]:
# sns.histplot(y_before, bins=80, kde=True, stat="density", label="before", color="blue", alpha=0.4)
# sns.histplot(y_after, bins=80, kde=True, stat="density", label="after", color="green", alpha=0.4)
# plt.legend(frameon=False)
# plt.xlabel("Mean daily milk production (kg)")
# plt.savefig("y_dist_before_after.png")

In [ ]:
import tqdm.auto
import tqdm
tqdm.auto.tqdm = tqdm.tqdm  # disable notebook widget, use plain text

In [ ]:
df_before, y, thi, t, idx, n_groups = prepare_train_df(df, '2013-01-01', '2016-01-01')
print("Before data: ", len(y), n_groups)
model_label = "whf_before__v7_lambda10"
trace_before, posterior_predictive_before = load_or_run_model(model_label, run_model, y, thi, t, idx, n_groups)
az.plot_posterior(trace_before,var_names=['mu_log_a','mu_b','mu_c', 'mu_gamma'])

In [ ]:
print(len(df_before), np.percentile(df_before.groupby("idx").count()['name'],[5,50,95]))

In [ ]:
df_after, y, thi, t, idx, n_groups = prepare_train_df(df, '2017-01-01', '2020-01-01')
print("After data: ", len(y),n_groups)
model_label = "whf_after_v7_lambda10"
trace_after, posterior_predictive_after = load_or_run_model(model_label, run_model, y, thi, t, idx, n_groups)
az.plot_posterior(trace_after,var_names=['mu_log_a','mu_b','mu_c', 'mu_gamma'])

In [ ]:
print(len(df_after), np.percentile(df_after.groupby("idx").count()['name'],[5,50,95]))

In [ ]:
with pm.Model() as model:

    # --- Hyperpriors ---
    mu_log_a = pm.Normal("mu_log_a", mu=3.5, sigma=0.5)
    sigma_log_a = pm.HalfNormal("sigma_log_a", sigma=0.3)

    mu_b = pm.Normal("mu_b", mu=0.2, sigma=0.1)
    sigma_b = pm.HalfNormal("sigma_b", sigma=0.1)

    mu_c = pm.HalfNormal("mu_c", sigma=0.05)
    sigma_c = pm.HalfNormal("sigma_c", sigma=0.01)

    #mu_beta = pm.Normal("mu_beta", mu=0, sigma=0.1)
    #sigma_beta = pm.HalfNormal("sigma_beta", sigma=0.05)

    mu_gamma = pm.Normal("mu_gamma", mu=0, sigma=0.01)
    sigma_gamma = pm.HalfNormal("sigma_gamma", sigma=0.005)

    # --- Group-level parameters (non-centered) ---
    log_a_offset = pm.Normal("log_a_offset", mu=0, sigma=1, shape=n_groups)
    log_a_group = pm.Deterministic("log_a_cycle", mu_log_a + log_a_offset * sigma_log_a)

    b_offset = pm.Normal("b_offset", mu=0, sigma=1, shape=n_groups)
    b_group = pm.Deterministic("b_cycle", mu_b + b_offset * sigma_b)

    c_offset = pm.Normal("c_offset", mu=0, sigma=1, shape=n_groups)
    c_group = pm.Deterministic("c_cycle", mu_c + c_offset * sigma_c)

    gamma_offset = pm.Normal("gamma_offset", mu=0, sigma=1, shape=n_groups)
    gamma_group = pm.Deterministic("gamma_cycle", mu_gamma + gamma_offset * sigma_gamma)

    #beta_offset = pm.Normal("beta_offset", mu=0, sigma=1, shape=n_groups)
    #beta_group = pm.Deterministic("beta_group", mu_beta + beta_offset * sigma_beta)

    # --- Observation noise ---
    sigma = pm.HalfNormal("sigma", sigma=1)

    # --- Model prediction ---
    log_a = log_a_group[idx]
    b = b_group[idx] #+ beta_group[idx] * thi
    c = c_group[idx] + gamma_group[idx] * thi  # group-specific gamma applied

    log_mu = log_a + b * pm.math.log(t) - c * t

    # --- Likelihood ---
    y_obs = pm.StudentT("y_obs", nu=2, mu=log_mu, sigma=sigma, observed=np.log(y))

graph = pm.model_to_graphviz(model)
graph.render("model_graph", format="pdf", cleanup=True)
# produces model_graph.png in the current directory

In [ ]:
plt.hist(df_after['weighted_THI'])

In [ ]:
plt.hist(df_after.loc[(df_after['Lactation Days']>100) & (df_after['Lactation Days']<250) & (df_after['weighted_THI']>0), 'weighted_THI'])

In [ ]:
sns.histplot(df_before['Day Production'], bins=60, kde=True, stat="density", label="before", color="blue", alpha=0.4)
sns.histplot(df_after['Day Production'], bins=60, kde=True, stat="density", label="after", color="green", alpha=0.4)
plt.legend(frameon=False)
plt.xlabel("Mean daily milk production (kg)")
plt.savefig("y_dist_before_after.pdf")

In [ ]:
sns.scatterplot(x=df_after['weighted_THI'], y=np.log1p(df_after['Day Production'].values), color='green', s=20,alpha=0.3, label='after')
sns.scatterplot(x=df_before['weighted_THI'], y=np.log1p(df_before['Day Production'].values), color='blue', s=20, alpha=0.3, label='before')
plt.xlabel("Experienced THI degrees")
plt.ylabel("log(Milk Yield)")
plt.legend(frameon=False)
plt.savefig("y_vs_thi_before_after.pdf")

In [ ]:
sns.histplot(df_before.loc[df_before['weighted_THI']>0,'weighted_THI'], bins=40, kde=True, stat="density", label="before", color="blue", alpha=0.4)
sns.histplot(df_after.loc[df_after['weighted_THI']>0,'weighted_THI'], bins=40, kde=True, stat="density", label="after", color="green", alpha=0.4)
plt.legend(frameon=False)
plt.xlabel("THI Degrees")
#plt.savefig("y_dist_before_after.png")

In [ ]:
summary = az.summary(trace_before, var_names=['mu_log_a', 'mu_b', 'mu_c', 'mu_gamma'], round_to=10)
mu_log_a_samples = trace_before.posterior['mu_log_a'].values  
mu_a_samples = np.exp(mu_log_a_samples)
idata_mu_a = az.from_dict(posterior={"mu_a": mu_a_samples})
summary_mu_a = az.summary(idata_mu_a, var_names=['mu_a'],round_to=10)
summary_before = pd.concat([summary, summary_mu_a])[['mean', 'sd', 'hdi_3%', 'hdi_97%']]

In [ ]:
summary = az.summary(trace_after, var_names=['mu_log_a', 'mu_b', 'mu_c', 'mu_gamma'], round_to=10)
mu_log_a_samples = trace_after.posterior['mu_log_a'].values  
mu_a_samples = np.exp(mu_log_a_samples)
idata_mu_a = az.from_dict(posterior={"mu_a": mu_a_samples})
summary_mu_a = az.summary(idata_mu_a, var_names=['mu_a'],round_to=10)
summary_after = pd.concat([summary, summary_mu_a])[['mean', 'sd', 'hdi_3%', 'hdi_97%']]

In [ ]:
# Format numbers: scientific for mu_gamma, fixed for others
def format_value(val, sci=False):
    if sci:
        return f"{val:.2e}"
    else:
        return f"{val:.4f}"

def format_summary(df):
    formatted = {}
    for param in df.index:
        sci = (param == 'mu_gamma')
        formatted[param] = [
            format_value(df.loc[param, 'mean'], sci),
            format_value(df.loc[param, 'sd'], sci),
            format_value(df.loc[param, 'hdi_3%'], sci),
            format_value(df.loc[param, 'hdi_97%'], sci),
        ]
    return pd.DataFrame.from_dict(
        formatted, orient='index', 
        columns=['Mean','SD','HDI 3%','HDI 97%']
    )

table_before = format_summary(summary_before)
table_after  = format_summary(summary_after)

# Combine side by side
final_table = pd.concat(
    {"Before": table_before, "After": table_after}, 
    axis=1
)

print(final_table)

In [ ]:
# Label the columns
summary_before['condition'] = 'before'
summary_after['condition']  = 'after'

# Combine into one table
summary_combined = pd.concat([summary_before, summary_after])

# Pivot for side-by-side comparison
summary_pivot = summary_combined.reset_index().pivot(index='index', columns='condition')
print(summary_pivot)

In [ ]:
# Extract posterior samples
param_plot = "mu_log_a"
posterior_before = np.exp(trace_before.posterior[param_plot].values.flatten())
posterior_after = np.exp(trace_after.posterior[param_plot].values.flatten())

# Plot
plt.figure(figsize=(10, 5))

# Prior
#plt.plot(x, prior_pdf, 'k--', label="Prior N(-0.01, 0.01²)")

# Posterior KDEs
sns.kdeplot(posterior_before, label="Posterior (Before Intervention)", fill=True, color="blue")
sns.kdeplot(posterior_after, label="Posterior (After Intervention)", fill=True, color="green")
#plt.xlim([-0.02,0.02])

# Labels and legend
#plt.title("Prior and Posterior Distributions for "+ param_plot)
plt.xlabel('mu_a')
plt.ylabel("Density")
plt.xlabel("a (Kg)")
plt.legend(frameon=False)
plt.grid(True)
plt.tight_layout()
plt.savefig('posteriors_a.pdf', dpi=300)
plt.show()

In [ ]:
# Extract posterior samples
param_plot = "mu_b"
posterior_before = trace_before.posterior[param_plot].values.flatten()
posterior_after = trace_after.posterior[param_plot].values.flatten()

# Plot
plt.figure(figsize=(10, 5))

# Prior
#plt.plot(x, prior_pdf, 'k--', label="Prior N(-0.01, 0.01²)")

# Posterior KDEs
sns.kdeplot(posterior_before, label="Posterior (Before Intervention)", fill=True, color="blue")
sns.kdeplot(posterior_after, label="Posterior (After Intervention)", fill=True, color="green")
#plt.xlim([-0.02,0.02])

# Labels and legend
#plt.title("Prior and Posterior Distributions for "+ param_plot)
plt.xlabel(param_plot)
plt.ylabel("Density")
plt.xlabel("b")
plt.legend(frameon=False)
plt.grid(True)
plt.tight_layout()
plt.savefig('posteriors_b.pdf', dpi=300)
plt.show()

In [ ]:
# Extract posterior samples
param_plot = "mu_c"
posterior_before = trace_before.posterior[param_plot].values.flatten()
posterior_after = trace_after.posterior[param_plot].values.flatten()

# Plot
plt.figure(figsize=(10, 5))

# Prior
#plt.plot(x, prior_pdf, 'k--', label="Prior N(-0.01, 0.01²)")

# Posterior KDEs
sns.kdeplot(posterior_before, label="Posterior (Before Intervention)", fill=True, color="blue")
sns.kdeplot(posterior_after, label="Posterior (After Intervention)", fill=True, color="green")
#plt.xlim([-0.02,0.02])

# Labels and legend
#plt.title("Prior and Posterior Distributions for "+ param_plot)
plt.xlabel(param_plot)
plt.ylabel("Density")
plt.xlabel("c (1/day)")
plt.legend(frameon=False)
plt.grid(True)
plt.tight_layout()
plt.savefig('posteriors_c.pdf', dpi=300)
plt.show()

In [ ]:
# Extract posterior samples
param_plot = "mu_gamma"
posterior_before = trace_before.posterior[param_plot].values.flatten()
posterior_after = trace_after.posterior[param_plot].values.flatten()

# Plot
plt.figure(figsize=(10, 5))

# Prior
#plt.plot(x, prior_pdf, 'k--', label="Prior N(-0.01, 0.01²)")

# Posterior KDEs
sns.kdeplot(posterior_before, label="Before Intervention", fill=True, color="blue")
sns.kdeplot(posterior_after, label="After Intervention", fill=True, color="green")
#plt.xlim([-0.02,0.02])

# Labels and legend
#plt.title("Prior and Posterior Distributions for "+ param_plot)
plt.xlabel(param_plot)
plt.ylabel("Posterior Density")
plt.xlabel(r'$\gamma$ (1/(°C·h·day))')
plt.legend(frameon=False)
plt.grid(True)
plt.tight_layout()
plt.savefig('posteriors.pdf', dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = [
    ("mu_log_a", "a (Kg)",                    lambda x: np.exp(x)),
    ("mu_b",     "b",                          lambda x: x),
    ("mu_c",     "c (1/day)",                  lambda x: x),
    ("mu_gamma", r"$\gamma$ (1/(°C·h·day))",   lambda x: x),
]

for ax, (param, xlabel, transform) in zip(axes.flatten(), params):
    before = transform(trace_before.posterior[param].values.flatten())
    after  = transform(trace_after.posterior[param].values.flatten())
    sns.kdeplot(before, label="Before Intervention", fill=True, color="blue",  ax=ax)
    sns.kdeplot(after,  label="After Intervention",  fill=True, color="green", ax=ax)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("Posterior Density", fontsize=12)
    if param=="mu_log_a":
        ax.legend(frameon=False)
    ax.grid(True)

for ax in axes[:, 1]:
    ax.set_ylabel("")
plt.tight_layout()
fig.savefig("posteriors_combined.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
rng = np.random.default_rng(42)
before = trace_before.posterior["mu_gamma"].values.flatten()
after = trace_after.posterior["mu_gamma"].values.flatten()

# Draw independent random pairs
before_sample = rng.choice(before, 100000, replace=True)
after_sample = rng.choice(after, 100000, replace=True)
prob = np.mean(before_sample > after_sample)
print(f"P(gamma_before > gamma_after) = {prob:.4f}")

# Correct sigma separation using marginal statistics
mean_diff = before.mean() - after.mean()
pooled_std = np.sqrt(before.std()**2 + after.std()**2)
z = mean_diff / pooled_std
print(f"Sigma separation = {z:.2f}")

In [ ]:
before = trace_before.posterior["mu_gamma"].values.flatten()
after = trace_after.posterior["mu_gamma"].values.flatten()

fig, ax = plt.subplots()
ax.hist(before, bins=100, alpha=0.5, density=True, label='before')
ax.hist(after, bins=100, alpha=0.5, density=True, label='after')
ax.legend()

# Also print the ranges explicitly
print(f"Before: mean={before.mean():.6e}, std={before.std():.6e}")
print(f"Before: 2.5th={np.percentile(before,2.5):.6e}, 97.5th={np.percentile(before,97.5):.6e}")
print(f"After:  mean={after.mean():.6e},  std={after.std():.6e}")
print(f"After:  2.5th={np.percentile(after,2.5):.6e},  97.5th={np.percentile(after,97.5):.6e}")

In [ ]:
from scipy import stats

mean_diff = before.mean() - after.mean()
std_diff = np.sqrt(before.std()**2 + after.std()**2)
prob_analytic = stats.norm.cdf(mean_diff / std_diff)
print(f"Analytical P(before > after) = {prob_analytic:.6f}")

In [ ]:
print(np.percentile(posterior_after/posterior_before,[2.5, 50, 97.5]))

In [ ]:
# Compute sigma difference
delta = trace_after.posterior["mu_gamma"].values.flatten() - trace_before.posterior["mu_gamma"].values.flatten()
z_score = np.abs(np.mean(delta)) / np.std(delta)

print(f"The posterior difference is approximately {z_score:.2f}σ")

In [ ]:
print(df_after['weighted_THI'].max())
print(np.log10(df_after['weighted_THI'].max()))

In [ ]:
# Extract posterior samples
posterior_pre = trace_before.posterior
posterior_post = trace_after.posterior

# Define THI input
thi = np.insert(np.logspace(1, 2.3, 10),0,0)
t_in = 205

# Compute predictions using posterior samples (PRE)
log_a_pre = posterior_pre["mu_log_a"].values.flatten()[:, np.newaxis]
b_pre = posterior_pre["mu_b"].values.flatten()[:, np.newaxis]
c_pre = posterior_pre["mu_c"].values.flatten()[:, np.newaxis]
gamma_pre = posterior_pre["mu_gamma"].values.flatten()[:, np.newaxis]

log_y_pred_samples_pre = (
    log_a_pre +
    b_pre * np.log(t_in) -
    (c_pre + gamma_pre * thi[np.newaxis, :] )* t_in
)
y_pred_samples_pre = np.exp(log_y_pred_samples_pre)

# Compute statistics: median, 1-sigma, and 2-sigma intervals
y_pred_pre_mean = np.mean(y_pred_samples_pre, axis=0)
y_pred_pre_1sigma = np.percentile(y_pred_samples_pre, [16, 84], axis=0)
y_pred_pre_2sigma = np.percentile(y_pred_samples_pre, [2.5, 97.5], axis=0)

# Same for POST
log_a_post = posterior_post["mu_log_a"].values.flatten()[:, np.newaxis]
b_post = posterior_post["mu_b"].values.flatten()[:, np.newaxis]
c_post = posterior_post["mu_c"].values.flatten()[:, np.newaxis]
gamma_post = posterior_post["mu_gamma"].values.flatten()[:, np.newaxis]

log_y_pred_samples_post = (
    log_a_post +
    b_post * np.log(t_in) -
    (c_post + gamma_post * thi[np.newaxis, :]) * t_in
)
y_pred_samples_post = np.exp(log_y_pred_samples_post)

# Compute statistics: median, 1-sigma, and 2-sigma intervals
y_pred_post_mean = np.mean(y_pred_samples_post, axis=0)
y_pred_post_1sigma = np.percentile(y_pred_samples_post, [16, 84], axis=0)
y_pred_post_2sigma = np.percentile(y_pred_samples_post, [2.5, 97.5], axis=0)

# Print or return the results as needed
print("y_pred_pre (mean):", y_pred_pre_mean)
print("1-sigma CI (pre):", y_pred_pre_1sigma)
print("2-sigma CI (pre):", y_pred_pre_2sigma)

print("y_pred_post (mean):", y_pred_post_mean)
print("1-sigma CI (post):", y_pred_post_1sigma)
print("2-sigma CI (post):", y_pred_post_2sigma)

# Optional: percent difference in means
percent_diff = 100 * (y_pred_post_mean - y_pred_pre_mean) / y_pred_pre_mean
print("Percent difference (post vs pre):", percent_diff)

In [ ]:
thi

In [ ]:
# Normalize predictions by baseline (first column), sample-wise
rel_y_pred_samples_pre = 100 * (y_pred_samples_pre / y_pred_samples_pre[:, [0]])
rel_y_pred_samples_post = 100 * (y_pred_samples_post / y_pred_samples_post[:, [0]])

# Compute mean and intervals
rel_y_pred_pre_mean = np.mean(rel_y_pred_samples_pre, axis=0)
rel_y_pred_pre_1sigma = np.percentile(rel_y_pred_samples_pre, [16, 84], axis=0)
rel_y_pred_pre_2sigma = np.percentile(rel_y_pred_samples_pre, [2.5, 97.5], axis=0)

rel_y_pred_post_mean = np.mean(rel_y_pred_samples_post, axis=0)
rel_y_pred_post_1sigma = np.percentile(rel_y_pred_samples_post, [16, 84], axis=0)
rel_y_pred_post_2sigma = np.percentile(rel_y_pred_samples_post, [2.5, 97.5], axis=0)

plt.figure(figsize=(10, 6))

# Convert THI back from log1p for x-axis
thi_vals = thi
print(pd.DataFrame({'thi': thi,'diff':rel_y_pred_pre_mean - rel_y_pred_post_mean}))

# --- PRE ---
plt.plot(thi_vals, rel_y_pred_pre_mean, '-s',label="Pre (samples)", color="blue")
plt.fill_between(thi_vals, rel_y_pred_pre_1sigma[0], rel_y_pred_pre_1sigma[1],
                 color="blue", alpha=0.2, label="Pre ±1σ")
plt.fill_between(thi_vals, rel_y_pred_pre_2sigma[0], rel_y_pred_pre_2sigma[1],
                   color="blue", alpha=0.15, label="Pre ±2σ")

# --- POST ---
plt.plot(thi_vals, rel_y_pred_post_mean, '-s',label="Post (samples)", color="green")
plt.fill_between(thi_vals, rel_y_pred_post_1sigma[0], rel_y_pred_post_1sigma[1],
                 color="green", alpha=0.2, label="Post ±1σ")
plt.fill_between(thi_vals, rel_y_pred_post_2sigma[0], rel_y_pred_post_2sigma[1],
                   color="green", alpha=0.15, label="Post ±2σ")

# plt.plot(thi_vals, 100*y_pred_pre/y_pred_pre[0], '-o', label='Pre (means)')
# plt.plot(thi_vals, 100*y_pred_post/y_pred_post[0],'-o', label='Post (means)')

# --- Labels and Layout ---
plt.xlabel("THI Cooling Degree Hours (°C·h)")
plt.ylabel("Predicted Yield (% of baseline)")
#plt.title("Relative Predicted Average Milk Yield vs THI (t = " + str(t_in) + ")")
plt.grid(True)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig('projections.pdf', dpi=300)
plt.show()

In [ ]:
# Group the data
grouped = df_after.groupby("idx")

# Select 100 posterior samples
posterior = trace_after.posterior.stack(sample=("chain", "draw"))
dataset_indices = posterior["log_a_group"].coords["log_a_group_dim_0"].values
posterior_samples = posterior.isel(sample=slice(0, 100))

# Function to compute Wood's model
def wood_curve(t, a, b, c):
    return a * (t ** b) * np.exp(-c * t)

# Build traces
dropdown_buttons = []
fig = go.Figure()

n_datasets = len(dataset_indices)
n_traces_per_dataset = 3  # data points, median line, shaded area
total_traces = n_datasets * n_traces_per_dataset

for i, dataset_idx in enumerate(dataset_indices):
    df_i = grouped.get_group(i)
    t = df_i["Lactation Days"].values
    y = df_i["Day Production"].values
    
    t_sorted = np.sort(t)
    
    # Draw multiple posterior predictive curves
    a_samples = np.exp(posterior_samples["log_a_group"][i, :].values)
    b_samples = posterior_samples["b_group"][i, :].values
    c_samples = posterior_samples["c_group"][i, :].values

    y_preds = np.stack([
        wood_curve(t_sorted, a, b, c)
        for a, b, c in zip(a_samples, b_samples, c_samples)
    ])
    
    # Compute median and 90% credible interval
    y_med = np.median(y_preds, axis=0)
    y_low = np.percentile(y_preds, 5, axis=0)
    y_high = np.percentile(y_preds, 95, axis=0)

    # Plot data
    fig.add_trace(go.Scatter(x=t, y=y, mode="markers",
                             marker=dict(
                                        color='LightSkyBlue',
                                        size=5,
                                        line=dict(
                                            color='MediumPurple',
                                            width=2
                                        )),
                             name=f"Data idx={dataset_idx}", visible=(i == 0)))

    # Plot median fit line
    fig.add_trace(go.Scatter(x=t_sorted, y=y_med, mode="lines",
                             name=f"Fit idx={dataset_idx}", visible=(i == 0),
                             line=dict(color='green')))

    # Plot uncertainty band as filled area
    fig.add_trace(go.Scatter(
        x=np.concatenate([t_sorted, t_sorted[::-1]]),
        y=np.concatenate([y_low, y_high[::-1]]),
        fill="toself",
        fillcolor="rgba(0,100,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
        visible=(i == 0)
    ))

    # --- Create button for this dataset ---
    visible = [False] * total_traces
    visible[3 * i] = True       # data
    visible[3 * i + 1] = True   # median fit
    visible[3 * i + 2] = True   # uncertainty band

    button = dict(
        label=f"Dataset {dataset_idx}",
        method="update",
        args=[{"visible": visible},
              {"title": f"Milk Yield - Dataset {dataset_idx}"}],
    )
    dropdown_buttons.append(button)

# Add dropdown to figure
fig.update_layout(
    updatemenus=[dict(
        buttons=dropdown_buttons,
        direction="down",
        showactive=True,
        x=0.1,
        xanchor="left",
        y=1.1,
        yanchor="top"
    )],
    title="Milk Yield vs Lactation Days (Posterior Fit)",
    xaxis_title="Lactation Days",
    yaxis_title="Milk Yield (Day Production)"
)

fig.show()
#fig.write_html("lac_curve_fit.html")

In [ ]:
# Group the data
grouped = df_after.groupby("idx")

# Build traces
dropdown_buttons = []
fig = go.Figure()

n_datasets = len(dataset_indices)

# Build both traces (y and y2) for each dataset
for i, dataset_idx in enumerate(dataset_indices):
    df_i = grouped.get_group(i)
    t = df_i["Lactation Days"].values
    y = df_i["weighted_THI"].values
    y_0 = df_i["Daily THI Degree"].values
    y2 = df_i["cumsum_THI"].values

    visible = (i == 0)

    # First Y-axis trace
    fig.add_trace(go.Scatter(
        x=t, y=y, mode="lines",
        line=dict(color='LightSkyBlue', width=2),
        #marker=dict(color='LightSkyBlue', size=5,
        #            line=dict(color='MediumPurple', width=2)),
        name=f"Weighted THI",
        visible=visible,
        yaxis='y1'
    ))

    # First Y-axis trace
    fig.add_trace(go.Scatter(
        x=t, y=y_0, mode="lines",
        line=dict(color='magenta', width=2),
        #marker=dict(color='magenta', size=5,
        #            line=dict(color='magenta', width=2)),
        name=f"Daily THI Degrees",
        visible=visible,
        yaxis='y1'
    ))

    # Second Y-axis trace
    fig.add_trace(go.Scatter(
        x=t, y=y2, mode="lines",
        line=dict(color='orange', width=2),
        name=f"Cumulative THI",
        visible=visible,
        yaxis='y2'
    ))

# Dropdown buttons — need to control 2 traces per dataset now
for i, dataset_idx in enumerate(dataset_indices):
    visible = [False] * (3 * n_datasets)
    visible[3 * i] = True      # weighted_THI
    visible[3 * i + 1] = True  # Daily THI Degree
    visible[3 * i + 2] = True  # cumsum_THI


    button = dict(
        label=f"Dataset {dataset_idx}",
        method="update",
        args=[
            {"visible": visible},
            {"title": f"THI Metrics - Dataset {dataset_idx}"}
        ],
    )
    dropdown_buttons.append(button)

# Add layout with second y-axis
fig.update_layout(
    updatemenus=[dict(
        buttons=dropdown_buttons,
        direction="down",
        showactive=True,
        x=0.1,
        xanchor="left",
        y=1.1,
        yanchor="top"
    )],
    title="THI Metrics vs Lactation Days",
    xaxis_title="Lactation Days",
    yaxis=dict(title="THI Degrees", side='left'),
    yaxis2=dict(title="Cumulative THI", overlaying='y', side='right'),
)

fig.show()
fig.write_html("THI3.html")

In [ ]:
def definite_integral(b, c, lower=1, upper=300):
    factor = 1 / c**(b + 1)
    gamma_term = gamma(b + 1)
    lower_term = gammainc(b + 1, c * lower)
    upper_term = gammainc(b + 1, c * upper)
    return factor * gamma_term * (upper_term - lower_term)

In [ ]:
#definite_integral(b, c, lower=1, upper=300)

In [ ]:
trace_before.posterior['mu_b'].values.flatten().mean()

In [ ]:
print(trace_before.posterior['mu_gamma'].values.mean() - 3*trace_before.posterior['mu_gamma'].values.std())
print(trace_after.posterior['mu_gamma'].values.mean() + 3*trace_after.posterior['mu_gamma'].values.std())

In [ ]:
diff = trace_before.posterior['mu_gamma'].values.mean() - trace_after.posterior['mu_gamma'].values.mean()
combined_sigma = np.sqrt(
    trace_before.posterior['mu_gamma'].values.std()**2 +
    trace_after.posterior['mu_gamma'].values.std()**2
)
sigma_diff = diff / combined_sigma
print(sigma_diff)

In [ ]:
# --- Wood's model ---
def wood_curve(t, a, b, c):
    return a * (t ** b) * np.exp(-c * t)

# --- Function to get posterior samples and predictions ---
def get_predictions(df, trace, n_samples=100, n_curves=6, seed=None):
    if seed is not None:
        random.seed(seed)
        
    grouped = df.groupby("idx")
    posterior = trace.posterior.stack(sample=("chain", "draw"))
    dataset_indices = posterior["log_a_group"].coords["log_a_group_dim_0"].values
    posterior_samples = posterior.isel(sample=slice(0, n_samples))

    selected_indices = random.sample(list(dataset_indices), n_curves)

    curves = []
    for i in selected_indices:
        df_i = grouped.get_group(i)
        t = df_i["Lactation Days"].values
        y = df_i["Day Production"].values
        t_sorted = np.sort(t)

        a_samples = np.exp(posterior_samples["log_a_group"][i, :].values)
        b_samples = posterior_samples["b_group"][i, :].values
        c_samples = posterior_samples["c_group"][i, :].values

        y_preds = np.stack([
            wood_curve(t_sorted, a, b, c)
            for a, b, c in zip(a_samples, b_samples, c_samples)
        ])

        y_med = np.median(y_preds, axis=0)
        y_low = np.percentile(y_preds, 5, axis=0)
        y_high = np.percentile(y_preds, 95, axis=0)

        curves.append({
            "t": t,
            "y": y,
            "t_sorted": t_sorted,
            "y_med": y_med,
            "y_low": y_low,
            "y_high": y_high,
            "idx": i
        })

    return curves

# --- Get predictions for both before and after ---
curves_before = get_predictions(df_before, trace_before, n_curves=6, seed=43)
curves_after = get_predictions(df_after, trace_after, n_curves=6, seed=44)

# --- Combine both into one list ---
all_curves = curves_before + curves_after

# --- Plotting ---
fig, axes = plt.subplots(3, 4, figsize=(14, 9), sharex=True, sharey=True)
axes = axes.flatten()

for ax, curve in zip(axes, all_curves):
    ax.scatter(curve["t"], curve["y"], s=8, color="tab:blue", alpha=0.7)
    ax.plot(curve["t_sorted"], curve["y_med"], color="black", linewidth=1)
    ax.fill_between(curve["t_sorted"], curve["y_low"], curve["y_high"], color="gray", alpha=0.3)
    
    #ax.set_title(f"idx = {curve['idx']}", fontsize=9)
    ax.tick_params(axis='both', labelsize=8)

# --- Layout cleanup ---
for i, ax in enumerate(axes):
    if i % 4 != 0:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    else:
        ax.set_ylabel("Milk Yield", fontsize=9)
    if i < 8:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Lactation Days", fontsize=9)

plt.tight_layout(h_pad=0.5, w_pad=0.5)
plt.savefig("lac_curves_12_panels.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 1. Reset index to match PPC shape
df_after = df_after.reset_index(drop=True)

# 2. Extract posterior predictive samples
ppc_y = posterior_predictive_after.posterior_predictive["y_obs"]  # shape: (chain, draw, obs)
ppc_stacked = ppc_y.stack(sample=("chain", "draw"))  # shape: (obs, samples)

# 3. Select idx = 0 group
ids = [10,20,2,13,15,26]
ppc_plot_data_after = []
for i in ids:
    group = df_after[df_after["idx"] == i]
    t = group["Lactation Days"].values
    y = group["Day Production"].values
    thi = group['weighted_THI'].values
    data_idx = group.index.values  # these indices align with ppc_stacked rows
    
    # 4. Extract PPC samples for this group
    y_preds = np.exp(ppc_stacked[data_idx, :]) # shape: [n_points, n_samples]
    
    # 5. Compute summary stats across samples
    y_med = np.median(y_preds, axis=1)
    y_low = np.percentile(y_preds, 5, axis=1)
    y_high = np.percentile(y_preds, 95, axis=1)
    
    # 6. Sort for plotting
    sort_idx = np.argsort(t)
    t_sorted = t[sort_idx]
    y_med_sorted = y_med[sort_idx]
    y_low_sorted = y_low[sort_idx]
    y_high_sorted = y_high[sort_idx]
    ppc_plot_data_after.append([t, y, t_sorted, y_med_sorted, y_low_sorted, y_high_sorted,thi])

# 7. Plot
fig, axes = plt.subplots(3, 2, figsize=(10, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, j in enumerate(zip(axes, all_curves)):
    ax, curve = j
    ppc_plot_data_in = ppc_plot_data_after[i]
    
    all_thi = np.concatenate([ppc_plot_data_after[ii][6] for ii in range(len(ppc_plot_data_after))])
    thi_min, thi_max = all_thi.min(), all_thi.max()
    t, y, thi = ppc_plot_data_after[i][0], ppc_plot_data_after[i][1], ppc_plot_data_after[i][6]
    sc = ax.scatter(t, y, c=thi, cmap="hot", vmin=thi_min, vmax=thi_max, s=10, alpha=0.8)

    #ax.scatter(ppc_plot_data_in[0], ppc_plot_data_in[1], s=10, color="black", label="Observed")
    ax.plot(ppc_plot_data_in[2], ppc_plot_data_in[3], color="green", label="Posterior Predictive Median")
    ax.fill_between(ppc_plot_data_in[2], ppc_plot_data_in[4], ppc_plot_data_in[5], color="green", alpha=0.1, label="90% CI")
    ax.tick_params(axis='both', labelsize=12)
    plt.tight_layout(h_pad=0., w_pad=0.)
    

for i, ax in enumerate(axes):
    if i % 2 != 0:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    else:
        ax.set_ylabel("Milk Yield (kg)", fontsize=12)
    if i < 3:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Lactation Days", fontsize=12)
fig.colorbar(sc, ax=axes.ravel().tolist(), label="THI",location='bottom',shrink=0.9, aspect=30,pad=0.05)
fig.savefig("lac_curves_after.pdf", dpi=300, bbox_inches="tight")

In [ ]:
# 1. Reset index to match PPC shape
df_before = df_before.reset_index(drop=True)

# 2. Extract posterior predictive samples
ppc_y = posterior_predictive_before.posterior_predictive["y_obs"]  # shape: (chain, draw, obs)
ppc_stacked = ppc_y.stack(sample=("chain", "draw"))  # shape: (obs, samples)

# 3. Select idx = 0 group
ids = [10,2,2,13,15,16]
ppc_plot_data = []
for i in ids:
    group = df_before[df_before["idx"] == i]
    t = group["Lactation Days"].values
    y = group["Day Production"].values
    thi = group['weighted_THI'].values
    data_idx = group.index.values  # these indices align with ppc_stacked rows
    
    # 4. Extract PPC samples for this group
    y_preds = np.exp(ppc_stacked[data_idx, :]) # shape: [n_points, n_samples]
    
    # 5. Compute summary stats across samples
    y_med = np.median(y_preds, axis=1)
    y_low = np.percentile(y_preds, 5, axis=1)
    y_high = np.percentile(y_preds, 95, axis=1)
    
    # 6. Sort for plotting
    sort_idx = np.argsort(t)
    t_sorted = t[sort_idx]
    y_med_sorted = y_med[sort_idx]
    y_low_sorted = y_low[sort_idx]
    y_high_sorted = y_high[sort_idx]
    ppc_plot_data.append([t, y, t_sorted, y_med_sorted, y_low_sorted, y_high_sorted,thi])

# 7. Plot
fig, axes = plt.subplots(3, 2, figsize=(10, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, j in enumerate(zip(axes, all_curves)):
    ax, curve = j
    ppc_plot_data_in = ppc_plot_data[i]
    
    all_thi = np.concatenate([ppc_plot_data_after[ii][6] for ii in range(len(ppc_plot_data_after))])
    thi_min, thi_max = all_thi.min(), all_thi.max()
    t, y, thi = ppc_plot_data[i][0], ppc_plot_data[i][1], ppc_plot_data[i][6]
    sc = ax.scatter(t, y, c=thi, cmap="hot", vmin=thi_min, vmax=thi_max, s=10, alpha=0.8)

    #ax.scatter(ppc_plot_data_in[0], ppc_plot_data_in[1], s=10, color="black", label="Observed")
    ax.plot(ppc_plot_data_in[2], ppc_plot_data_in[3], color="blue", label="Posterior Predictive Median")
    ax.fill_between(ppc_plot_data_in[2], ppc_plot_data_in[4], ppc_plot_data_in[5], color="blue", alpha=0.1, label="90% CI")
    ax.tick_params(axis='both', labelsize=12)
    plt.tight_layout(h_pad=0., w_pad=0.)

for i, ax in enumerate(axes):
    if i % 2 != 0:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    else:
        ax.set_ylabel("Milk Yield (kg)", fontsize=12)
    if i < 3:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Lactation Days", fontsize=12)
fig.colorbar(sc, ax=axes.ravel().tolist(), label="THI",location='bottom',shrink=0.9, aspect=30,pad=0.05)
fig.savefig("lac_curves_before.pdf", dpi=300, bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12), sharex=True, sharey=True)

all_thi_combined = np.concatenate(
    [ppc_plot_data[ii][6] for ii in range(6)] +
    [ppc_plot_data_after[ii][6] for ii in range(6)]
)
thi_min, thi_max = all_thi_combined.min(), all_thi_combined.max()

datasets = [
    (ppc_plot_data,       "blue",  "Before"),
    (ppc_plot_data_after, "green", "After"),
]

for panel, (data, color, title) in enumerate(datasets):
    col_start = panel * 2
    for i, ax in enumerate(axes[:, col_start:col_start + 2].flatten()):
        t, y, thi_vals = data[i][0], data[i][1], data[i][6]
        sc = ax.scatter(t, y, c=thi_vals, cmap="hot", vmin=thi_min, vmax=thi_max, s=10, alpha=0.8)
        ax.plot(data[i][2], data[i][3], color=color)
        ax.fill_between(data[i][2], data[i][4], data[i][5], color=color, alpha=0.1)
        ax.tick_params(axis="both", labelsize=12)
    #axes[0, col_start].set_title(title, fontsize=14, fontweight="bold")

for row in range(3):
    axes[row, 0].set_ylabel("Milk Yield (kg)", fontsize=12)
for col in range(4):
    axes[2, col].set_xlabel("Lactation Days", fontsize=12)

plt.tight_layout(h_pad=0., w_pad=0.)
fig.colorbar(sc, ax=axes.ravel().tolist(), label="THI",
             location="bottom", shrink=0.9, aspect=60, pad=0.05)
fig.savefig("lac_curves_combined.pdf", dpi=300, bbox_inches="tight")

In [ ]:
diag_before = az.summary(
    trace_before,
    var_names=["mu_log_a", "mu_b", "mu_c", "mu_gamma"],
    round_to='none',
    stat_focus="median"
)
diag_after = az.summary(
    trace_after,
    var_names=["mu_log_a", "mu_b", "mu_c", "mu_gamma"],
    round_to='none',
    stat_focus="median"
)

for label, diag in [("Before", diag_before), ("After", diag_after)]:
    diag_display = diag.copy().astype(float)
    gamma_mask = diag_display.index.str.contains("gamma")
    scale_cols = [c for c in diag_display.columns if c not in ["ess_median", "ess_tail", "r_hat"]]
    #diag_display.loc[gamma_mask, scale_cols] = diag_display.loc[gamma_mask, scale_cols] * 1e6
    print(f"--- {label} ---")
    print(diag_display)

In [ ]:
def diag_to_latex(diag, caption, label):
    param_names = {
        'mu_log_a': r'$\mu_{\log a}$',
        'mu_b':     r'$\mu_b$',
        'mu_c':     r'$\mu_c$',
        'mu_gamma': r'$\mu_\gamma$',
    }
    col_names = {
        'median':      'Median',
        'mad':         'MAD',
        'eti_3%':      r'ETI $3\%$',
        'eti_97%':     r'ETI $97\%$',
        'mcse_median': 'MCSE',
        'ess_median':  'ESS (med.)',
        'ess_tail':    'ESS (tail)',
        'r_hat':       r'$\hat{R}$',
    }
    df = diag.copy().astype(float)
    df.index = [param_names.get(x, x) for x in df.index]
    df.columns = [col_names.get(x, x) for x in df.columns]

    def fmt_sci(x):  return f'{x:.2e}'
    def fmt_ess(x):  return f'{int(round(x)):,}'
    def fmt_rhat(x): return f'{x:.2f}'

    formatters = {
        col: (fmt_ess if col in ('ESS (med.)', 'ESS (tail)')
              else fmt_rhat if col == r'$\hat{R}$'
              else fmt_sci)
        for col in df.columns
    }

    col_fmt = 'l' + 'r' * len(df.columns)
    body = df.to_latex(
        formatters=formatters,
        escape=False,
        column_format=col_fmt,
    )
    # Strip the bare tabular wrapper so we can wrap it in table
    body_lines = body.split('\n')
    inner = '\n'.join(body_lines[1:-2])  # drop \begin{tabular}...\end{tabular} wrappers

    table = (
        f'\\begin{{table}}[htbp]\n'
        f'\\centering\n'
        f'\\small\n'
        f'\\caption{{{caption}}}\n'
        f'\\label{{{label}}}\n'
        + inner +
        '\n\\end{table}'
    )
    return table


before_caption = (
    r'MCMC convergence diagnostics -- before treatment. '
    r'\textit{Median} and \textit{MAD} are the posterior median and median absolute deviation. '
    r'ETI = equal-tailed credible interval (3\%--97\%). '
    r'MCSE = Monte Carlo standard error of the median. '
    r'ESS = effective sample size. '
    r'$\hat{R}$ = potential scale reduction factor; values $<1.01$ indicate convergence.'
)
after_caption = before_caption.replace('before treatment', 'after treatment')

print(diag_to_latex(diag_before, before_caption, 'tab:diag_before'))
print()
print(diag_to_latex(diag_after, after_caption, 'tab:diag_after'))


In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 6),
    gridspec_kw={'hspace': 0.4, 'wspace': 0.3}
)

# Plot before
az.plot_trace(
    trace_before,
    var_names=["mu_gamma"],
    compact=False,
    combined=False,
    axes=axes[0:1, :],
    show=False
)

# Plot after
az.plot_trace(
    trace_after,
    var_names=["mu_gamma"],
    compact=False,
    combined=False,
    axes=axes[1:2, :],
    show=False
)

# Remove all auto-generated titles
for ax in axes.flatten():
    ax.set_title("")

# Set x-labels
axes[0, 0].set_xlabel(r"$\mu_\gamma$", fontsize=12)
axes[1, 0].set_xlabel(r"$\mu_\gamma$", fontsize=12)
axes[0, 1].set_xlabel("Draw", fontsize=12)
axes[1, 1].set_xlabel("Draw", fontsize=12)

# Add before/after text annotations inside the plots
axes[0, 0].text(
    0.97, 0.95, "Before intervention",
    transform=axes[0, 0].transAxes,
    fontsize=11,
    ha='right', va='top',
    bbox=dict(boxstyle='round,pad=0.3', 
              facecolor='white', 
              edgecolor='gray', 
              alpha=0.8)
)
axes[1, 0].text(
    0.97, 0.95, "After intervention",
    transform=axes[1, 0].transAxes,
    fontsize=11,
    ha='right', va='top',
    bbox=dict(boxstyle='round,pad=0.3', 
              facecolor='white', 
              edgecolor='gray', 
              alpha=0.8)
)

# Mirror the annotation on the trace panels
axes[0, 1].text(
    0.97, 0.95, "Before intervention",
    transform=axes[0, 1].transAxes,
    fontsize=11,
    ha='right', va='top',
    bbox=dict(boxstyle='round,pad=0.3', 
              facecolor='white', 
              edgecolor='gray', 
              alpha=0.8)
)
axes[1, 1].text(
    0.97, 0.95, "After intervention",
    transform=axes[1, 1].transAxes,
    fontsize=11,
    ha='right', va='top',
    bbox=dict(boxstyle='round,pad=0.3', 
              facecolor='white', 
              edgecolor='gray', 
              alpha=0.8)
)

plt.savefig("trace_mu_gamma.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
az.summary(trace_before,var_names=['mu_log_a','mu_b','mu_c', 'mu_gamma'])

In [ ]:
az.summary(trace_after,var_names=['mu_log_a','mu_b','mu_c', 'mu_gamma'])

In [ ]:
gammas_before = []
for i in df_before['idx'].unique():
    gammas_before.append(trace_before.posterior['gamma_group'].values[:,:,i].flatten())

gammas_after = []
for i in df_after['idx'].unique():
    gammas_after.append(trace_after.posterior['gamma_group'].values[:,:,i].flatten())

In [ ]:
for i in gammas_before:
    sns.kdeplot(i,fill=False,alpha=0.1,color='blue')
sns.kdeplot(trace_before.posterior['mu_gamma'].values.flatten(),color='blue',fill=True, alpha=1)

for i in gammas_after:
    sns.kdeplot(i,fill=False,alpha=0.1,color='green')
sns.kdeplot(trace_after.posterior['mu_gamma'].values.flatten(),color='green', fill=True, alpha=1)

In [ ]:
a_samples = trace_before.posterior['mu_log_a'].values.flatten()
b_samples = trace_before.posterior['mu_b'].values.flatten()
c_samples = trace_before.posterior['mu_c'].values.flatten()
gamma_samples = trace_before.posterior['mu_gamma'].values.flatten()

# Stack into matrix
samples = np.vstack([a_samples, b_samples, c_samples, gamma_samples])

# Compute correlation matrix
cor_matrix = np.corrcoef(samples)

print(cor_matrix)

labels = ['a', 'b', 'c', 'γ']
plt.figure(figsize=(6, 5))
sns.heatmap(cor_matrix, xticklabels=labels, yticklabels=labels, 
            annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8})
#plt.title("Correlation Matrix: Farm-level Parameters")
plt.tight_layout()
plt.savefig("corr_matrix_before.pdf", dpi=300, bbox_inches="tight")
plt.show()

az.plot_pair(trace_before,var_names=['mu_log_a','mu_b','mu_c','mu_gamma'],kind='kde')

In [ ]:
a_samples = trace_after.posterior['mu_log_a'].values.flatten()
b_samples = trace_after.posterior['mu_b'].values.flatten()
c_samples = trace_after.posterior['mu_c'].values.flatten()
gamma_samples = trace_after.posterior['mu_gamma'].values.flatten()

# Stack into matrix
samples = np.vstack([a_samples, b_samples, c_samples, gamma_samples])

# Compute correlation matrix
cor_matrix = np.corrcoef(samples)

labels = ['a', 'b', 'c', 'γ']
plt.figure(figsize=(6, 5))
sns.heatmap(cor_matrix, xticklabels=labels, yticklabels=labels, 
            annot=True, fmt=".2f", cmap='Greens', vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8})
#plt.title("Correlation Matrix: Farm-level Parameters")
plt.tight_layout()
plt.savefig("corr_matrix_after.pdf", dpi=300, bbox_inches="tight")
plt.show()

print(cor_matrix)
az.plot_pair(trace_after,var_names=['mu_log_a','mu_b','mu_c','mu_gamma'],kind='kde')

In [ ]:
# --- Combined correlation matrix: before & after ---
labels = [r'$\log a$', r'$b$', r'$c$', r'$\gamma$']

def get_cor_matrix(trace):
    samples = np.vstack([
        trace.posterior['mu_log_a'].values.flatten(),
        trace.posterior['mu_b'].values.flatten(),
        trace.posterior['mu_c'].values.flatten(),
        trace.posterior['mu_gamma'].values.flatten(),
    ])
    return np.corrcoef(samples)

cor_before = get_cor_matrix(trace_before)
cor_after  = get_cor_matrix(trace_after)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

hm_kw = dict(
    xticklabels=labels, yticklabels=labels,
    annot=True, fmt='.2f', vmin=-1, vmax=1,
    square=True, cbar_kws={'shrink': 0.8},
)

sns.heatmap(cor_before, ax=axes[0], cmap='Blues',  **hm_kw)
sns.heatmap(cor_after,  ax=axes[1], cmap='Greens', **hm_kw)

plt.tight_layout()
plt.savefig('corr_matrix_combined.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
y_avg = []
for i in range(len(a_samples)): # mcmc. 
    total_milk = 0.0
    for j in df_after['idx'].unique():
        sel = df_after['idx'] == j
        days = df_after.loc[sel, 'Lactation Days'].values
        THI = df_after.loc[sel, 'weighted_THI'].values

        # Log milk yield
        y_hat_log = (a_samples[i] +
                     b_samples[i] * np.log(days) -
                     (c_samples[i] + gamma_samples[i] * THI) * days)
        y_hat = np.exp(y_hat_log)

        total_milk += y_hat.sum()
    
    y_avg.append(total_milk)

In [ ]:
y_avg_cf = []
gamma_samples_cf = trace_before.posterior['mu_gamma'].values.flatten()
for i in range(len(a_samples)):
    total_milk = 0.0
    for j in df_after['idx'].unique():
        sel = df_after['idx'] == j
        days = df_after.loc[sel, 'Lactation Days'].values
        THI = df_after.loc[sel, 'weighted_THI'].values

        # Log milk yield
        y_hat_log = (a_samples[i] +
                     b_samples[i] * np.log(days) -
                     (c_samples[i] + gamma_samples_cf[i] * THI) * days)
        y_hat = np.exp(y_hat_log)

        total_milk += y_hat.sum()
    
    y_avg_cf.append(total_milk)

In [ ]:
print("Unique animals before", len(df_before['name'].unique()))
print("Unique cycles before", len(df_before['idx'].unique()))
print("Unique animals after", len(df_after['name'].unique()))
print("Unique cycles after", len(df_after['idx'].unique()))

In [ ]:
pd.merge(
    df_before.assign(name=df_before['name'].astype(int))[["name","idx"]],
    df_after.assign(name=df_after['name'].astype(int))[["name","idx"]],
    on="name",
    how="inner",
    suffixes=("_before","_after")
)

In [ ]:
y_avg = np.array(y_avg)
y_avg_cf = np.array(y_avg_cf)

print(np.median(y_avg))
print(df_after['Day Production'].sum())
print(100*(np.median(y_avg)-df_after['Day Production'].sum())/df_after['Day Production'].sum())
y_avg_percentiles = np.percentile(y_avg,[2.5,50, 97.5])

In [ ]:
unique_cycles = len(df_after['idx'].unique())*1000
sns.histplot(y_avg/unique_cycles, color='green', alpha=0.3, stat='density', kde=True, label='Posterior MY')
plt.axvspan(
    y_avg_percentiles[0]/unique_cycles,  # 10th percentile
    y_avg_percentiles[-1]/unique_cycles,  # 90th percentile
    color='green',
    alpha=0.1,
    label='95% CI'
)
plt.axvline(x=df_after['Day Production'].sum()/unique_cycles, color='black', linewidth=2, label='Observed MY')
plt.axvline(x=y_avg_percentiles[1]/unique_cycles, linestyle='--', color='green', linewidth=2, label='Model Median')
plt.xlabel("Milk yield (tonnes per lactation cycle)")
plt.legend(frameon=False)
plt.savefig("total_after_my.pdf", dpi=300, bbox_inches="tight")

In [ ]:
# counterfactula comparison
100*np.percentile((y_avg - y_avg_cf)/y_avg, [2.5, 50, 97.5])

In [ ]:
2.65399528 - 1.32156891
#3.96757614 - 2.65399528

In [ ]:
ATE_samples = y_avg - y_avg_cf
print(np.mean(ATE_samples), np.std(ATE_samples))
ATE_samples_percentiles = np.percentile(ATE_samples, [2.5, 50, 97.5])
print(ATE_samples_percentiles/unique_cycles)

sns.histplot(ATE_samples/unique_cycles, kde=True, color='darkorange',stat='density', label='Marginal Effect')

plt.axvspan(
    ATE_samples_percentiles[0]/unique_cycles,  
    ATE_samples_percentiles[-1]/unique_cycles,
    color='darkorange',
    alpha=0.1,
    label='95% CI'
)

plt.xlabel("Marginal effect (tonnes per lactation cycle)")
plt.legend(frameon=False)
plt.savefig("marginal_effect.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
milk_cost = 0.6 # euro ??
no_cows = 50
cooling_system_cost = 3000 # upfront
costs_percentiles = np.percentile(ATE_samples*milk_cost,[2.5, 97.5])

In [ ]:
100*np.percentile(ATE_samples/y_avg, [2.5,50,97.5])

In [ ]:
sns.histplot(ATE_samples*milk_cost/1000, kde=True, color='darkorange',stat='density',label='Marginal Effect')
plt.axvspan(
    costs_percentiles[0]/1000,  
    costs_percentiles[-1]/1000,
    color='darkorange',
    alpha=0.1,
    label='95% CI'
)
plt.axvline(x=3,linestyle='-', color='green',linewidth=2, label='Initial cost')
plt.xlabel("Marginal effect (x1000 €)")
plt.legend(frameon=False)
plt.savefig("marginal_effect_euros.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# --- Combined figure: posterior MY (top) + marginal effects (bottom) ---
fig = plt.figure(figsize=(12, 9))
gs  = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.3)
ax_top = fig.add_subplot(gs[0, 0])   # top: full width
ax_bl  = fig.add_subplot(gs[1, 0])   # bottom-left
ax_br  = fig.add_subplot(gs[1, 1])   # bottom-right

# --- (a) Posterior milk yield ---
sns.histplot(y_avg / unique_cycles, ax=ax_top,
             color='green', alpha=0.3, stat='density', kde=True, label='Posterior MY')
ax_top.axvspan(y_avg_percentiles[0] / unique_cycles,
               y_avg_percentiles[-1] / unique_cycles,
               color='green', alpha=0.1, label='95% CI')
ax_top.axvline(x=df_after['Day Production'].sum() / unique_cycles,
               color='black', linewidth=2, label='Observed MY')
ax_top.axvline(x=y_avg_percentiles[1] / unique_cycles,
               linestyle='--', color='green', linewidth=2, label='Model Median')
ax_top.set_xlabel('Milk yield (tonnes per lactation cycle)')
ax_top.legend(frameon=False)

# --- (b) Marginal effect (tonnes) ---
sns.histplot(ATE_samples / unique_cycles, ax=ax_bl,
             kde=True, color='darkorange', stat='density', label='Marginal Effect')
ax_bl.axvspan(ATE_samples_percentiles[0] / unique_cycles,
              ATE_samples_percentiles[-1] / unique_cycles,
              color='darkorange', alpha=0.1, label='95% CI')
ax_bl.set_xlabel('Marginal effect (tonnes per lactation cycle)')
ax_bl.legend(frameon=False)

# --- (c) Marginal effect (euros) ---
sns.histplot(ATE_samples * milk_cost / 1000, ax=ax_br,
             kde=True, color='darkorange', stat='density', label='Marginal Effect')
ax_br.axvspan(costs_percentiles[0] / 1000,
              costs_percentiles[-1] / 1000,
              color='darkorange', alpha=0.1, label='95% CI')
ax_br.axvline(x=3, linestyle='-', color='green', linewidth=2, label='Initial cost')
ax_br.set_xlabel(r'Marginal effect ($\times$1000 \euro)')
ax_br.legend(frameon=False)

# # Panel labels
# for ax, lbl in [(ax_top, '(a)'), (ax_bl, '(b)'), (ax_br, '(c)')]:
#     ax.text(-0.07, 1.04, lbl, transform=ax.transAxes,
#             fontsize=12, fontweight='bold', va='bottom')

plt.savefig('results_combined.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Pick individual index (e.g., cow 0)
idx = 53

# Slice the individual cow's parameters
data_ind = {
    'log_a': trace_after.posterior['log_a_group'][:, 0],
    'log_b': trace_after.posterior['b_group'][:, idx],
    'log_c': trace_after.posterior['c_group'][:, idx],
    'gamma': trace_after.posterior['gamma_group'][:, idx],
}

# Now plot
az.plot_pair(data_ind, 
             kind='kde', 
             marginals=True, 
             colorbar='blues', 
             kde_kwargs={"quantiles":[0.1, 0.3, 0.5, 0.7, 0.9], "contourf_kwargs": {"colors":None, "cmap": "Blues"}})

In [ ]:
df_after.idx.unique()

In [ ]:
plt.hist(df_after[['idx','Day Production']].groupby('idx').mean(),10)

In [ ]:
np.percentile(df_after.loc[df_after.weighted_THI>0,'weighted_THI'].values,[2.5,50,97.5])

In [ ]:
plt.hist(df_after.loc[df_after.weighted_THI>0,'weighted_THI'].values)

In [ ]:
percentileofscore(df_after.loc[df_after.weighted_THI>0,'weighted_THI'].values, 100)

In [ ]:
max(df_before['Daily THI Degree']*6/24)

In [ ]:
idx_to_check = []
i = 0
for cycle_in in [3,4]:
    for idx in df.loc[sel_0, 'Animal Name'].unique():
        sel = (
        (df['Animal Name'] == idx) &
        (df['Lactation Number']==cycle_in) &
        (df['Lactation Days'] < 305) &
        (df['Lactation Days'] > 1) &
        (df['Day Production'] > 15) &
        (~df['Day Production'].isna()) &
        (df['DateTime'] > pd.to_datetime('2013-01-01')) &
        (df['DateTime'] < pd.to_datetime('2016-01-01'))
        )
        df2 = df.loc[sel,['DateTime','Lactation Days','Day Production', 'Daily THI Degree']].drop_duplicates() #.groupby(['DateTime','Lactation Days'],as_index=False).sum()
        if len(df2)<=150: 
            idx_to_check.append(idx)
            continue
            